In [1]:
from pyspark.sql.functions import col, when, lit, current_date, datediff

StatementMeta(, 4136eb37-e38e-4d0f-af71-ac2ef7d234cf, 3, Finished, Available, Finished, False)

In [2]:
# 1. LOAD BRONZE DATA
df_bronze = spark.table("bronze_assets")

StatementMeta(, 4136eb37-e38e-4d0f-af71-ac2ef7d234cf, 4, Finished, Available, Finished, False)

In [8]:
# 2. DATA CLEANING & STANDARDIZATION
# We ensure types are correct and filter out any obvious garbage (if any)
df_clean = df_bronze.filter(col("AssetID").isNotNull())
display(df_clean)

StatementMeta(, 4136eb37-e38e-4d0f-af71-ac2ef7d234cf, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37d42073-9df0-4e67-9016-14ab11af6ae3)

In [9]:
# 3. CALCULATE ASSET HEALTH SCORE (The "Omexom" Business Logic)
# We start with 100 points and subtract based on risks:
# - High Temp (>75C): -30 points
# - Low Oil (<50%): -40 points
# - High Load (>85%): -10 points
# - Age (>15 years): -10 points

df_silver = df_clean.withColumn(
    "Health_Score",
    lit(100) - 
    when(col("Temperature_C") > 75, 30).otherwise(0) -
    when(col("OilLevel_Pct") < 50, 40).otherwise(0) -
    when(col("Load_Pct") > 85, 10).otherwise(0)
)
display(df_silver)

StatementMeta(, 4136eb37-e38e-4d0f-af71-ac2ef7d234cf, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 82028943-43e3-4634-aa61-c2e11c00bf5e)

In [10]:
# 4. ASSIGN HEALTH STATUS
df_silver = df_silver.withColumn(
    "Health_Status",
    when(col("Health_Score") >= 80, "Healthy")
    .when(col("Health_Score") >= 60, "Monitor")
    .otherwise("Critical")
)
display(df_silver)

StatementMeta(, 4136eb37-e38e-4d0f-af71-ac2ef7d234cf, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 14419876-05f8-4a8a-b284-faeeb51cb8e0)

In [11]:
# 5. SAVE TO SILVER LAYER
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_asset_health")

print("✅ Success! Silver layer created with Health Scoring logic.")

# 6. SHOW THE ANOMALIES WE CAUGHT
# This will show you exactly which assets are failing
display(df_silver.filter(col("Health_Status") == "Critical").select("AssetID", "AssetType", "Temperature_C", "OilLevel_Pct", "Health_Score"))

StatementMeta(, 4136eb37-e38e-4d0f-af71-ac2ef7d234cf, 13, Finished, Available, Finished, False)

✅ Success! Silver layer created with Health Scoring logic.


SynapseWidget(Synapse.DataFrame, 5415db42-b825-4a86-aee2-802c6ffb327b)